# 01 - Google Trends Ingestion

## Purpose
This notebook will be used for Google Trends extraction for the Coffee Dashboard project.

## Current status
Paused for now, as the project currently used a saved CSV file as input for Power BI development.

---

In [130]:
import pandas as pd
from pathlib import Path

In [131]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Interim data folder:", INTERIM_DIR)

Project root: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard
Raw data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw
Interim data folder: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim


In [132]:
RAW_FILE = RAW_DIR / "google_trends_batch_01.csv"

print("Raw file exists:", RAW_FILE.exists())
print("Raw file path:", RAW_FILE)

Raw file exists: True
Raw file path: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\raw\google_trends_batch_01.csv


In [133]:
df_raw = pd.read_csv(RAW_FILE)
df_raw.head()

,date,iced coffee,iced latte,cappuccino,espresso,cold brew,isPartial
0,2021-04-04,7,2,10,32,8,False
1,2021-04-11,8,2,10,32,8,False
2,2021-04-18,8,2,10,34,9,False
3,2021-04-25,7,2,10,31,8,False
4,2021-05-02,7,2,10,32,8,False


In [134]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         262 non-null    str  
 1   iced coffee  262 non-null    int64
 2   iced latte   262 non-null    int64
 3   cappuccino   262 non-null    int64
 4   espresso     262 non-null    int64
 5   cold brew    262 non-null    int64
 6   isPartial    262 non-null    bool 
dtypes: bool(1), int64(5), str(1)
memory usage: 12.7 KB


In [135]:
df_raw.columns.tolist()

['date',
 'iced coffee',
 'iced latte',
 'cappuccino',
 'espresso',
 'cold brew',
 'isPartial']

In [136]:
df_raw.columns = [col.strip() for col in df_raw.columns]

if "isPartial" in df_raw.columns:
    df_raw = df_raw.drop(columns=["isPartial"])

df_raw.head()

,date,iced coffee,iced latte,cappuccino,espresso,cold brew
0,2021-04-04,7,2,10,32,8
1,2021-04-11,8,2,10,32,8
2,2021-04-18,8,2,10,34,9
3,2021-04-25,7,2,10,31,8
4,2021-05-02,7,2,10,32,8


---

In [137]:
df_fact = df_raw.melt(
    id_vars="date",
    var_name="keyword",
    value_name="search_interest"
)

df_fact["date"] = pd.to_datetime(df_fact["date"])
df_fact["search_interest"] = pd.to_numeric(df_fact["search_interest"])

df_fact = df_fact.sort_values(["keyword", "date"]).reset_index(drop=True)

df_fact.head

<bound method NDFrame.head of            date     keyword  search_interest
0    2021-04-04  cappuccino               10
1    2021-04-11  cappuccino               10
2    2021-04-18  cappuccino               10
3    2021-04-25  cappuccino               10
4    2021-05-02  cappuccino               10
...         ...         ...              ...
1305 2026-03-08  iced latte                3
1306 2026-03-15  iced latte                4
1307 2026-03-22  iced latte                4
1308 2026-03-29  iced latte                4
1309 2026-04-05  iced latte                4

[1310 rows x 3 columns]>

In [138]:
df_fact.info()

<class 'pandas.DataFrame'>
RangeIndex: 1310 entries, 0 to 1309
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             1310 non-null   datetime64[us]
 1   keyword          1310 non-null   str           
 2   search_interest  1310 non-null   int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 30.8 KB


In [139]:
OUTPUT_FILE = INTERIM_DIR / "google_trends_fact.csv"
df_fact.to_csv(OUTPUT_FILE, index=False)

print(f"Saved fact table to: {OUTPUT_FILE}")

Saved fact table to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\google_trends_fact.csv


In [140]:
# Run checks on the fact table

# Basic structure
print("Fact table shape:", df_fact.shape)
print("\nColumns:")
print(df_fact.columns.tolist())

# Null checks
print("\nNull values by column:")
print(df_fact.isna().sum())

# Duplicate checks
duplicate_count = df_fact.duplicated().sum()
print("\nNumber of duplicated rows:", duplicate_count)

# Check keyword list
print("\nKeywords:")
print(sorted(df_fact["keyword"].dropna().unique()))

# Check date range
print("\nMin date:", df_fact["date"].min())
print("Max date:", df_fact["date"].max())

# Search interest range
print("\nMinimum search interest:", df_fact["search_interest"].min())
print("Maximum search interest:", df_fact["search_interest"].max())

Fact table shape: (1310, 3)

Columns:
['date', 'keyword', 'search_interest']

Null values by column:
date               0
keyword            0
search_interest    0
dtype: int64

Number of duplicated rows: 0

Keywords:
['cappuccino', 'cold brew', 'espresso', 'iced coffee', 'iced latte']

Min date: 2021-04-04 00:00:00
Max date: 2026-04-05 00:00:00

Minimum search interest: 1
Maximum search interest: 100


---

In [141]:
# Create the date dimension table

dim_date = pd.DataFrame({
    "date": pd.to_datetime(df_fact["date"].drop_duplicates().sort_values())
})

dim_date["year"] = dim_date["date"].dt.year
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.day_name()

dim_date.head()

,date,year,month_number,month_name,day_of_month,day_name
0,2021-04-04,2021,4,April,4,Sunday
1,2021-04-11,2021,4,April,11,Sunday
2,2021-04-18,2021,4,April,18,Sunday
3,2021-04-25,2021,4,April,25,Sunday
4,2021-05-02,2021,5,May,2,Sunday


In [142]:
dim_date.info()

<class 'pandas.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          262 non-null    datetime64[us]
 1   year          262 non-null    int32         
 2   month_number  262 non-null    int32         
 3   month_name    262 non-null    str           
 4   day_of_month  262 non-null    int32         
 5   day_name      262 non-null    str           
dtypes: datetime64[us](1), int32(3), str(2)
memory usage: 9.3 KB


In [143]:
DATE_DIM_FILE = INTERIM_DIR / "dim_date.csv"
dim_date.to_csv(DATE_DIM_FILE, index=False)

print(f"Saved date dimension to: {DATE_DIM_FILE}")
print("Shape:", dim_date.shape)

Saved date dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_date.csv
Shape: (262, 6)


---

In [144]:
# Create the trend dimension table (to improve)

dim_trend = pd.DataFrame({
    "keyword": df_fact["keyword"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Mapping
trend_metadata = {
    "cappuccino": {"drink_category": "coffee", "trend_type": "classic"},
    "cold brew": {"drink_category": "coffee", "trend_type": "classic"},
    "espresso": {"drink_category": "coffee", "trend_type": "classic"},
    "iced coffee": {"drink_category": "coffee", "trend_type": "classic"},
    "iced latte": {"drink_category": "coffee", "trend_type": "classic"}
}

dim_trend["drink_category"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("drink_category"))
dim_trend["trend_type"] = dim_trend["keyword"].map(lambda x: trend_metadata.get(x, {}).get("trend_type"))

dim_trend.head()

,keyword,drink_category,trend_type
0,cappuccino,coffee,classic
1,cold brew,coffee,classic
2,espresso,coffee,classic
3,iced coffee,coffee,classic
4,iced latte,coffee,classic


In [145]:
TREND_DIM_FILE = INTERIM_DIR / "dim_trend.csv"
dim_trend.to_csv(TREND_DIM_FILE, index=False)

print(f"Saved trend dimension to: {TREND_DIM_FILE}")

Saved trend dimension to: C:\Users\lisan\OneDrive\Bureau\work\Projects\coffee-dashboard\data\interim\dim_trend.csv


---

In [146]:
# Paused for now

# from pytrends.request import TrendReq
#
# keywords = [
#     "iced coffee",
#     "iced latte",
#     "cappuccino",
#     "espresso",
#     "cold brew",
# ]
#
# pytrends = TrendReq(hl="en-US", tz=360)
# pytrends.build_payload(
#     kw_list=keywords,
#     timeframe="today 5-y",
#     geo=""
# )
#
# df_trends = pytrends.interest_over_time()
# df_trends.head()
#
# output_file = RAW_DIR / "google_trends_batch_01.csv"
# df_trends.to_csv(output_file)
# print(f"Saved to {output_file}")